# 04 · Agentic RAG: let the model decide what to search

**You'll learn:** what changes when RAG becomes an *agent*, how this project's `core/agentic_rag.py` does it in about 150 lines, when it beats classic RAG, what it costs, and a checklist for converting your own RAG pipeline.

**Time:** about 10 minutes · **Cost:** about 5¢ · **Runs on its own:** the second cell builds any missing data.

| Part | What it covers |
|---|---|
| 1 | Classic vs agentic RAG |
| 2 | The building blocks: tool, decision format, loop, guardrails |
| 3 | Watch the agent work |
| 4 | Classic vs agentic, side by side |
| 5 | How to convert a RAG pipeline into an agent |
| 6 | Things to try |

---
## Setup

### Setup
The next cell connects the notebook to the project. Read its first lines: `USE_EMBEDDED_QDRANT` picks where the database runs (Docker server, or embedded with no Docker). It prints which LLM, embedding model and database this run uses, all taken from your `.env`.

In [2]:
# ── Settings ────────────────────────────────────────────────────────────
# False: use the Qdrant server from `docker compose up -d`.
# True:  no Docker. Qdrant runs inside this notebook and stores data in data/qdrant_local/.
#        Restart the kernel after changing this, and run one notebook at a time
#        (only one process can open the embedded folder).
USE_EMBEDDED_QDRANT = False

# Setup: run from the repo root so `core`/`ingestion` imports and data/ paths work.
import logging
import os
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
logging.basicConfig(level=logging.WARNING)
if USE_EMBEDDED_QDRANT:
    os.environ["QDRANT_PATH"] = "data/qdrant_local"
warnings.filterwarnings("ignore", message="Qdrant client version")  # harmless client/server version notice

import pandas as pd

import core  # noqa: F401  (loads .env)
import core.embeddings as emb
import core.llm as llm
from core.clients import COLLECTION_NAME, QDRANT_PATH, QDRANT_URL, get_qdrant_client

pd.set_option("display.max_colwidth", 120)
print(f"LLM:        {llm.PROVIDER} / {llm.MODEL}")
print(f"Embeddings: {emb.PROVIDER} / {emb.MODEL}")
print(f"Qdrant:     {'embedded @ ' + QDRANT_PATH if QDRANT_PATH else 'server @ ' + QDRANT_URL}")

LLM:        gemini / gemini-3.5-flash
Embeddings: gemini / gemini-embedding-001
Qdrant:     server @ http://localhost:6333


In [3]:
# Is the vector database running?
try:
    names = [c.name for c in get_qdrant_client().get_collections().collections]
    print("Qdrant is up. Collections:", names or "(none yet)")
except Exception as exc:
    raise SystemExit(
        "Qdrant is not reachable. Either open Docker Desktop and run `docker compose up -d`, "
        "or set USE_EMBEDDED_QDRANT = True in the first cell and restart the kernel."
    ) from exc

# Make sure the data is ready, so this notebook runs on its own. prepare_data() fetches,
# chunks and indexes only what's missing: a few minutes the very first time, then seconds.
from ingestion.prepare import prepare_data

for name, count in prepare_data().items():
    print(f"{name}: {count} points")

Qdrant is up. Collections: ['sec_filings_advanced', 'sec_filings_advanced_gemini', 'sec_filings_basic', 'sec_filings_basic_gemini']
sec_filings_basic_gemini: 168 points
sec_filings_advanced_gemini: 310 points


---
## Part 1 · Classic vs agentic RAG

**Classic RAG** (notebooks 02–03) searches **once**, with the user's exact words, and answers from whatever comes back:

```mermaid
flowchart LR
  Q["Question"] --> S["Search once<br/>with the question"] --> P["Prompt + LLM"] --> A["Answer"]
```

That breaks on questions with several parts ("compare X with Y, and tell me Z") or questions whose words don't match the filing's words. **Agentic RAG** lets the LLM drive the search: it decides what to search for, reads the results, and searches again until it has enough or runs out of budget:

```mermaid
flowchart TD
  Q["Question"] --> T{"LLM decides<br/>next action"}
  T -->|"search(query)"| S["hybrid_search(query)"]
  S --> R["Results added<br/>to the transcript"]
  R --> T
  T -->|"answer + cited ids"| V{"Cited ids were<br/>actually retrieved?"}
  V -->|"yes"| A["Answer + citations"]
  V -->|"no"| N["'Not found'"]
```

| | Classic (`core/rag.py`) | Agentic (`core/agentic_rag.py`) |
|---|---|---|
| Who writes the search query | the user's question, as is | the LLM, one focused query at a time |
| Searches per question | 1 | 1 to 4, decided by the LLM |
| LLM calls per question | 1 | one per decision (searches + 1) |
| Multi-part questions | one search must cover every part | one search per part |
| Cost and latency | lowest | higher, grows with each search |
| Grounding and refusal | cite retrieved chunks, else "not found" | the same rules |

---
## Part 2 · The building blocks

### 2.1 The tool

The agent has exactly one tool: the same `hybrid_search()` the classic pipeline uses, returning 5 chunks per search. An agent is only as good as its tools; nothing about search itself changes.

### 2.2 The decision format

Each turn, the LLM must reply with **one JSON object**: either `{"action": "search", ...}` or `{"action": "answer", ...}`. Plain JSON, instead of a provider's native tool-calling API, keeps the same code working with OpenAI and Gemini. These are the real instructions:

In [4]:
from core.agentic_rag import AGENT_INSTRUCTIONS, MAX_SEARCHES, SEARCH_TOP_K

print(AGENT_INSTRUCTIONS.format(max_searches=MAX_SEARCHES))
print(f"\n(each search returns {SEARCH_TOP_K} chunks)")

You answer questions about companies' SEC 10-K filings. You cannot see the
filings directly; you have one tool, a search over filing excerpts.

Reply with exactly ONE JSON object and nothing else, in one of these two forms:
{"action": "search", "query": "<a focused search query>", "reason": "<why this search>"}
{"action": "answer", "answer": "<your answer>", "cited": [<excerpt ids you used>]}

Rules:
- Answer ONLY from excerpts returned by your searches. Cite the [id] of every excerpt you use.
- Split multi-part questions into separate, focused searches (one topic per search).
- If a search result doesn't cover what you need, search again with different words.
- If the excerpts don't contain the answer, say so and reply with "cited": [].
- If two searches in a row find nothing relevant, stop searching and answer with "cited": [].
- Never repeat a search you already made.
- You have at most 4 searches. Searches left are shown below.

(each search returns 5 chunks)


### 2.3 The loop

`answer_question_agentic()` keeps a running **transcript**: the question, then every search and its results. Each turn it:

1. sends the instructions + transcript + "searches left: N" to the LLM;
2. reads the JSON decision (a small parser tolerates code fences and extra text);
3. on **search**: runs `hybrid_search(query)`, appends the results to the transcript, and goes again;
4. on **answer**: keeps only cited ids that were really retrieved, and returns the answer with citations.

### 2.4 Guardrails

| Risk | Guardrail in the code |
|---|---|
| Endless searching | at most `max_searches` (4); after that the agent must answer |
| Repeating itself | a search it already made is refused, without using up the budget |
| Citing something it never saw | cited ids are checked against the chunks it actually retrieved |
| Answering from memory | no valid citation → the same "not found" message as classic RAG |
| Broken reply | invalid JSON → it is asked again; the loop has a hard turn limit |

---
## Part 3 · Watch the agent work

A two-part question: margins live in one place in the filing, the geographic decline in another. The trace shows each search the agent chose and why.

In [5]:
from core.agentic_rag import answer_question_agentic

AGENT_QUESTION = (
    "Compare Products gross margin to Services gross margin in 2024, "
    "and say which geographic segment had the largest net sales decline."
)
run = answer_question_agentic(AGENT_QUESTION)

display(pd.DataFrame([
    {"search": i + 1, "query the agent wrote": s.query, "why": s.reason, "chunks found": len(s.excerpt_ids)}
    for i, s in enumerate(run.steps)
]))
print("ANSWER:", run.result.answer, "\n")
for c in run.result.citations:
    print(f"- {c.section[:80]}")
print(f"\n{run.result.input_tokens} tokens in, {run.result.output_tokens} out, ≈ ${run.result.cost_usd:.4f}")

Embedding with Gemini:   0%|          | 0/1 [00:00<?, ?chunk/s]

,search,query the agent wrote,why,chunks found
0,1,Products Services net sales cost of sales gross margin 2024 geographic segment net sales,"Search for segment performance, Products vs Services revenue and cost of sales/gross margin in 2024, and geographic ...",5


ANSWER: In 2024, Apple's Products gross margin was $109,633 million (or 37.2% gross margin percentage), compared to Services gross margin of $71,050 million (or 73.9% gross margin percentage). Thus, Services had a significantly higher gross margin percentage than Products, while Products generated a higher dollar amount of gross margin.

Among the reportable geographic segments, Greater China had the largest net sales decline in 2024, decreasing by $5,607 million from $72,559 million in 2023 to $66,952 million in 2024 (all other geographic segments grew in net sales during 2024). 

- Item 7.    Management's Discussion and Analysis of Financial Condition and Resul
- Item 8.    Financial Statements and Supplementary Data > Note 13 - Segment Infor

2707 tokens in, 275 out, ≈ $0.0015


**What to notice:** the agent rewrote the question into short, specific queries, one per topic, and each citation comes from a search it made.

---
## Part 4 · Classic vs agentic, side by side

Three kinds of question: a simple one, a multi-part one, and one the filing can't answer.

In [6]:
import time

from core.rag import answer_question

questions = {
    "simple": "Why did Greater China net sales decrease in fiscal 2024?",
    "multi-part": AGENT_QUESTION,
    "not in filing": "What was Apple's revenue in fiscal year 2019?",
}

rows = []
for kind, q in questions.items():
    start = time.time()
    classic = answer_question(q)
    rows.append({"question": kind, "pipeline": "classic", "searches": 1, "citations": len(classic.citations),
                 "seconds": round(time.time() - start, 1), "cost_usd≈": round(classic.cost_usd, 4), "answer": classic.answer[:160]})
    start = time.time()
    agentic = answer_question_agentic(q)
    rows.append({"question": kind, "pipeline": "agentic", "searches": len(agentic.steps), "citations": len(agentic.result.citations),
                 "seconds": round(time.time() - start, 1), "cost_usd≈": round(agentic.result.cost_usd, 4), "answer": agentic.result.answer[:160]})
pd.DataFrame(rows)

Embedding with Gemini:   0%|          | 0/1 [00:00<?, ?chunk/s]

KeyboardInterrupt: 

**How to read it:** on a simple question the agent often makes one search and matches classic RAG at about the same cost. Each extra search adds an LLM call, so cost and time grow with the number of searches (roughly 2× on a two-part question). It earns that on multi-part or loosely-worded questions. Both refuse the out-of-scope question. A common production pattern is to use classic RAG by default and route only complex questions to the agent.

---
## Part 5 · How to convert a RAG pipeline into an agent

The steps this project took, in order. Each maps to code in `core/agentic_rag.py`:

1. **Turn retrieval into a tool.** Keep your existing search function (`hybrid_search`) and give the LLM a way to call it with its own query.
2. **Define the decision format.** A small, strict menu of actions (here `search` and `answer`), returned as JSON. Native tool calling (OpenAI tools, Gemini function calling) is the provider-specific alternative.
3. **Write the loop.** Prompt → decision → run the tool → add the results to the transcript → repeat.
4. **Keep an evidence store.** Remember every chunk retrieved (`seen`), so citations can be validated against it.
5. **Keep your guardrails.** The search budget, refusing repeats, validating citations, and the same "not found" rule as before.
6. **Log the trace.** Return the steps (`AgentRun.steps`) so you can see why it answered what it did.
7. **Measure both.** Run classic and agentic on the same golden set and compare quality, latency and cost before switching.

```mermaid
flowchart LR
  A["Classic RAG"] --> B["1 · Retrieval as a tool"] --> C["2 · Decision format"] --> D["3 · Loop"] --> E["4–5 · Evidence + guardrails"] --> F["6–7 · Trace + measure"] --> G["Agentic RAG"]
```

**Natural next tools** to add to the menu: look up a section by name (`Item 1C`), a calculator for percentages, or a second collection (another company or year).

---
## Part 6 · Things to try

- Ask a question that needs **three** facts and watch how many searches it makes.
- Lower the budget: `answer_question_agentic(AGENT_QUESTION, max_searches=1)`. Does the answer get worse?
- Point the agent at the basic chunks: `answer_question_agentic(AGENT_QUESTION, collection_name="sec_filings_basic")`.

In [ ]:
run = answer_question_agentic(AGENT_QUESTION, max_searches=1)
print([s.query for s in run.steps])
print(run.result.answer)